[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

/home/user/miniconda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [9]:
torch.arange(3).unsqueeze(1).shape

torch.Size([3, 1])

In [11]:
torch.arange(0, 10, 2)

tensor([0, 2, 4, 6, 8])

In [56]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    # pass
    B, S, D = q.shape
    pos = torch.arange(S, device=q.device).unsqueeze(1).float()
    assert pos.shape == (S, 1)
    dim = torch.arange(0, D, 2, device=q.device).float()
    assert dim.shape == (D // 2,)
    theta = 1 / 10000 ** (dim / D)
    assert theta.shape == (D // 2,)
    angles = pos * dim
    assert angles.shape == (S, D//2)
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)
    # print(angles.shape, angles)
    def rotate(x):
        x0, x1 = x[:, :, 0::2], x[:, :, 1::2]
        assert x0.shape[-1] == D // 2
        assert x0.shape[-2] == S
        assert x1.shape[-1] == D // 2
        assert x1.shape[-2] == S
        #print("new x0", x0*cos_a - x1*sin_a)
        #print("new x1", x0*sin_a + x1*cos_a)
        #print(torch.stack([x0*cos_a - x1*sin_a, x0*sin_a + x1*cos_a], dim=-1))
        return torch.stack([x0*cos_a - x1*sin_a, x0*sin_a + x1*cos_a], dim=-1).flatten(-2)
    return rotate(q), rotate(k)

In [57]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [58]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.6ms)
  ✅ [2/4] Preserves norm (2.1ms)
  ✅ [3/4] Relative position property (2.8ms)
  ✅ [4/4] Gradient flow (1.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (7.8ms total)
  Progress saved. Run status() to see your dashboard.

